In [77]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import numpy as np
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.fit_params.weibull_fit_params import WeibullFitParams

plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

In [78]:
power_law_db = pl.read_parquet(r"MLE_random_sample_fit_data\PowerLaw")
weibull_db = pl.read_parquet(r"MLE_random_sample_fit_data\Weibull")
log_normal_db = pl.read_parquet(r"MLE_random_sample_fit_data\LogNormal")

In [79]:
columns_to_select = ["aic", "numb_alphas"]

aic_db = pl.concat([
    power_law_db.select(columns_to_select).with_columns(pl.lit("PL").alias("fit_type")),
    weibull_db.select(columns_to_select).with_columns(pl.lit("W").alias("fit_type")),
    log_normal_db.select(columns_to_select).with_columns(pl.lit("LN").alias("fit_type"))
]).with_columns(
    (pl.col("aic") - pl.col("aic").min()).alias("delta_aic")
)

aic_db

aic,numb_alphas,fit_type,delta_aic
f64,i64,str,f64
2.5967e6,192734,"""PL""",1323.595424
2.6020e6,192734,"""PL""",6665.0313
2.5997e6,192734,"""PL""",4361.191384
2.6038e6,192734,"""PL""",8482.52188
2.5972e6,192734,"""PL""",1848.738895
…,…,…,…
2.6042e6,192734,"""LN""",8827.589764
2.6076e6,192734,"""LN""",12276.113486
2.5988e6,192734,"""LN""",3475.54321


In [80]:
# aic_db = aic_db.join(
#         aic_db.group_by("numb_alphas").agg(pl.col("aic").min().alias("min_aic_for_numb_alphas")),
#         on = "numb_alphas",
#     ).with_columns(
#         (pl.col("aic") - pl.col("min_aic_for_numb_alphas")).alias("delta_aic")
#     )

# aic_db

In [81]:
aic_db.group_by("fit_type").agg(
    pl.col("delta_aic").mean().alias("rel_mean_aic"),
    pl.col("delta_aic").std().alias("std_aic"),
    pl.col("delta_aic").min().alias("rel_min_aic"),
    pl.col("delta_aic").max().alias("rel_max_aic"),
    pl.len()
)

fit_type,rel_mean_aic,std_aic,rel_min_aic,rel_max_aic,len
str,f64,f64,f64,f64,u32
"""PL""",7052.467955,3707.256348,505.309401,18617.534164,120
"""LN""",5880.681282,3679.50379,0.0,16038.0026,100
"""W""",6691.554835,4016.738864,291.106226,28373.446026,340


In [83]:
import numpy as np
import polars as pl

types = aic_db["fit_type"].unique().sort().to_list()

samples = {
    t: aic_db.filter(pl.col("fit_type") == t)["delta_aic"].to_numpy()
    for t in types
}

n_boot = 10000
rng = np.random.default_rng()

# Store bootstrap means
boot_means = {t: np.empty(n_boot) for t in types}

# Generate bootstrap means
for i in range(n_boot):
    for t, x in samples.items():
        boot = rng.choice(x, size=len(x), replace=True)
        boot_means[t][i] = boot.mean()

# Probability each mean is the smallest
wins = {t: 0 for t in types}

for i in range(n_boot):
    means = {t: boot_means[t][i] for t in types}
    winner = min(means, key=means.get)
    wins[winner] += 1

# Report results
for t in types:
    mean = boot_means[t].mean()
    se = boot_means[t].std(ddof=1)
    lo, hi = np.percentile(boot_means[t], [2.5, 97.5])

    print(
        f"{t:10s}: "
        f"mean = {mean:.3f} ± {se:.3f} "
        f"(95% CI [{lo:.3f}, {hi:.3f}]), "
        f"P(smallest) = {wins[t]/n_boot:.3%}"
    )

LN        : mean = 5885.270 ± 364.155 (95% CI [5180.906, 6607.357]), P(smallest) = 96.260%
PL        : mean = 7049.970 ± 337.440 (95% CI [6396.479, 7724.092]), P(smallest) = 0.730%
W         : mean = 6692.637 ± 217.430 (95% CI [6272.013, 7123.398]), P(smallest) = 3.010%
